# x3a_infracoes — Autos de Infração IBAMA por Georreferenciamento Espacial

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** autos de infração (IBAMA), quantidade, municipal pelas coordenadas, anual  
**Conceito e Mecanismo Econômico:** Contagem de autos atribuídos ao município a partir das coordenadas geográficas (lat/long) do auto, corrigindo distorções onde a sede foi informada em vez do local do dano.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__autos_ibama__espacial_municipio_anual_candidato.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - n_autos_ibama_espacial já está agregado; comparar cobertura e concordância com o código informado.
> - Variante de associação territorial dos mesmos registros; não somar os dois agregados.

### Salvaguarda Territorial (141 Municípios de Mato Grosso):
> **Regra Obrigatória do Projeto:** O estado de Mato Grosso é considerado com exatamente **141 municípios**.  
> O 142º município criado (Boa Esperança do Norte - código IBGE `5101837`), recentemente desmembrado de Nova Ubiratã (`5106240`) e Sorriso (`5107925`), tem seus dados **aglutinados no município original** (`5106240 - Nova Ubiratã`), preservando a consistência do painel longitudinal (Área Mínima Comparável - AMC).

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`codigo_ibge`**: Código do município determinado espacialmente (141 municípios).
- **`ano`**: Ano da autuação.
- **`n_autos_ibama_espacial`**: Número de autos atribuídos espacialmente.
- **`valor_multas_ibama_espacial`**: Valor das multas atribuídas espacialmente.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__autos_ibama__espacial_municipio_anual_candidato.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x3a_infracoes`:
- **Atenção:** n_autos_ibama_espacial já está agregado; comparar cobertura e concordância com o código informado.
- **Atenção:** Variante de associação territorial dos mesmos registros; não somar os dois agregados.
- **Salvaguarda Territorial:** Harmonização estrita para 141 municípios de Mato Grosso. Se houver dados de Boa Esperança do Norte (`5101837`), são aglutinados em Nova Ubiratã (`5106240`).


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['codigo_ibge'] = df['codigo_ibge'].astype(str).str.zfill(7)
df['ano'] = pd.to_numeric(df['ano'], errors='coerce')
df['n_autos_ibama_espacial'] = pd.to_numeric(df['n_autos_ibama_espacial'], errors='coerce')

# HARMONIZAÇÃO TERRITORIAL (141 MUNICÍPIOS):
df['codigo_ibge'] = df['codigo_ibge'].replace({'5101837': '5106240'})
df = df[df['codigo_ibge'].str.startswith('51')].copy()

# REGRA DO TXT: 'Variante de associação territorial dos mesmos registros; não somar os dois agregados.'
df = df.groupby(['codigo_ibge', 'ano'], as_index=False).agg({
    'n_autos_ibama_espacial': 'sum',
    'valor_multas_ibama_espacial': 'sum'
})
print(f"Municípios únicos com registros espaciais: {df['codigo_ibge'].nunique()}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva dos autos espaciais por município-ano:")
print(df['n_autos_ibama_espacial'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
sns.histplot(df[df['n_autos_ibama_espacial'] > 0]['n_autos_ibama_espacial'], bins=30, log_scale=True, color='teal')
plt.title("Distribuição dos Autos Espaciais (Escala Log)")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
sns.lineplot(data=df_painel, x='ano', y='n_autos_ibama_espacial', estimator='sum', color='teal')
plt.title("Total Anual de Autos Espaciais em MT (2001-2024)")
plt.show()

## Fase 6: Feature Engineering — Área Autuada por Infração Ambiental (Ibama)

### 1. Step 1: Agregação Anual da Área Autuada (ha)
- A área autuada reflete a magnitude física do desmatamento ilegal flagrado pelos fiscais.
- Grade balanceada de 141 municípios $\times$ 24 anos com zero verdadeiro para ausência de autuações.

### 2. Step 2: Derivações
- **Transformação $\ln(1 + AreaAutuada_{t-1})$:** Devido à assimetria extrema (a maioria dos municípios tem zero ha autuados, mas alguns atingem dezenas de milhares de hectares), a escala logarítmica é obrigatória.
- **Proporção da Área Nativa Autuada:** $\frac{AreaAutuada_{t-1}}{NativeVeg_{t-1}}$, mede o percentual da mata do município sob sanção.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Estruturação na Grade Balanceada 2001-2024
mun_mt = df['codigo_ibge'].unique()
grade_completa = pd.MultiIndex.from_product([mun_mt, range(2001, 2025)], names=['codigo_ibge', 'ano']).to_frame().reset_index(drop=True)

painel_area_aut = pd.merge(grade_completa, df, on=['codigo_ibge', 'ano'], how='left')

# Preenchimento de zeros verdadeiros para municípios sem autuação no ano
if 'n_autos_ibama_espacial' in painel_area_aut.columns:
    painel_area_aut['n_autos_espacial'] = painel_area_aut['n_autos_ibama_espacial'].fillna(0)

painel_area_aut.sort_values(['codigo_ibge', 'ano'], inplace=True)

# 2. Defasagem de 1 período (t-1)
painel_area_aut['n_autos_espacial_lag1'] = painel_area_aut.groupby('codigo_ibge')['n_autos_espacial'].shift(1)
painel_area_aut['ln_autos_espacial_lag1'] = np.log1p(painel_area_aut['n_autos_espacial_lag1'])

print("Painel de Autos Espacializados Pronto:")
print(painel_area_aut[['codigo_ibge', 'ano', 'n_autos_espacial', 'n_autos_espacial_lag1', 'ln_autos_espacial_lag1']].head())